# Developer 02 Simple Debugging (LangChain 2026)

## What This Lesson Is
Instrument chain execution with callback traces for debugging.

## Scientific Lens
- Concept: Trace-based debugging
- Measure: Trace completeness and failure diagnosability
- Validity Limit: Notebook traces are not full distributed tracing.


## How It Works
1. Build deterministic trace events.
2. Classify failure causes.
3. Run live callback-instrumented invoke.


In [ ]:
print('Debugging lesson preflight complete')


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: real provider/tool path with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
trace = [{'stage':'prompt','ok':True},{'stage':'invoke','ok':False,'error':'timeout'},{'stage':'retry','ok':True}]
print(trace)
assert any(t.get('error')=='timeout' for t in trace)


In [ ]:
# Live Demo
import os
from langchain.callbacks.base import BaseCallbackHandler

class TraceCallback(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        print('[trace] start', len(prompts))
    def on_llm_end(self, response, **kwargs):
        print('[trace] end')

try:
    from langchain_openai import ChatOpenAI
except Exception as exc:
    print(f"Skipping live run: langchain_openai unavailable ({exc})")
else:
    if not os.getenv("OPENAI_API_KEY"):
        print("Skipping live run: OPENAI_API_KEY not set.")
    else:
        llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0, callbacks=[TraceCallback()], timeout=20)
        out = llm.invoke('One sentence: why callback traces matter.')
        print(out.content if hasattr(out, 'content') else out)


## Applied Labs
1. Write trace events as JSON lines to file.
2. Add retry_count field and analyze failures.
3. Inject one simulated parsing failure and trace it.

## Validation Checklist
- Trace start/end hooks are visible.
- Failure events include explicit cause.
- Live run stays resilient with callback enabled.

## Further Reading
- [LangChain Callbacks](https://python.langchain.com/docs/concepts/callbacks/)
- [LangSmith](https://docs.smith.langchain.com/)
- [OpenTelemetry Primer](https://opentelemetry.io/docs/concepts/observability-primer/)
